# Hidden Markov Model from Scratch

- Gaussian emission model for daily log-returns
- Forward Algorithm
- Backward Algorithm
- Viterbi Algorithm
- Baum-Welch EM
- Regime analysis & interactive visualisations

Charts are interactive **Plotly** figures.

---

### References
1. Rabiner (1989). *A tutorial on hidden Markov models*
2. Nguyen & Nguyen (2015). *Hidden Markov Model for Stock Selection*, Risks 3(4)
3. McGreevy, J. (2021). *Hidden Markov Models in Finance*, Imperial College MSc
4. Paolucci, R. (2025). *HMMs for Quantitative Finance*, Quant Guild (reference PDF)
5. Chen, X. (2025). *HMM-based market regime detection with RL for portfolio management*, IDS

---

## Review note against standard regime-detection practice

The notebook follows the common research workflow used in financial regime detection: estimate latent states from returns, sort the resulting states by volatility for interpretability, inspect transition persistence, and compare probabilistic temporal models against static clustering or anomaly-detection baselines. HMMs are expected to show smoother and more persistent regimes because they explicitly estimate transition probabilities. GMM and K-Means are useful baselines, but their assignments are independent across time unless a separate smoothing or transition model is added. Isolation Forest is treated as a stress/anomaly detector rather than a complete bull-neutral-bear regime model.

## 0. Setup & Imports

In [ ]:
%pip install -q numpy pandas scipy plotly yfinance ipywidgets

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm
import warnings
warnings.filterwarnings("ignore")

# Project utilities
from utils.data_utils import prepare_ticker_data, get_observation_sequence, time_split
from utils.viz_utils  import (plot_price_with_regimes, plot_regime_distributions,
                               plot_transition_matrix, plot_alpha_beta, plot_gamma,
                               plot_log_likelihood, plot_viterbi_path)
from utils.metrics    import (regime_statistics, regime_persistence_summary,
                               conditional_sharpe, aic, bic, hmm_n_params)

# Our from-scratch implementation
from hmm_core import GaussianHMM, forward_pass, backward_pass, viterbi, compute_gamma, compute_xi

print("All imports OK")

## 1. Data - S&P 500 (SPY) Daily Log-Returns

We fetch daily OHLCV data and compute:
$$r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right)$$

The HMM in this notebook is trained on this one-dimensional return sequence. Volatility is interpreted later from the fitted state's return standard deviation rather than added as a separate input feature.

In [ ]:
TICKER = "SPY"
df = prepare_ticker_data(TICKER, start="2024-01-01")
df_train, df_test = time_split(df, train_frac=0.80)

# HMM input: keep this one-dimensional on purpose.
# A univariate model is easier to inspect and explain in an interview.
X_train = get_observation_sequence(df_train)
X_test  = get_observation_sequence(df_test)
X_all   = get_observation_sequence(df)

print(f"Total observations : {len(X_all)}")
print(f"Train             : {len(X_train)}  ({df_train.index[0].date()} - {df_train.index[-1].date()})")
print(f"Test              : {len(X_test)}   ({df_test.index[0].date()} - {df_test.index[-1].date()})")
print(f"\nReturn stats: mean={X_all.mean()*252:.2%}  ann.vol={X_all.std()*np.sqrt(252):.2%}")


In [ ]:
# Basic data sanity checks before fitting anything.
assert df_train.index.max() < df_test.index.min(), "Train/test split should respect time order"
assert np.isfinite(X_train).all() and np.isfinite(X_test).all(), "Returns should not contain NaN/inf"

print("Training return summary:")
print(df_train["Returns"].describe().round(6))


In [ ]:
# Interactive price + return chart
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("SPY Adjusted Close", "Daily Log-Return"))

fig.add_trace(go.Scatter(x=df.index, y=df["Close"],
    line=dict(color="rgba(150,180,255,0.9)", width=1.2), name="Close"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["Returns"],
    line=dict(color="rgba(99,200,180,0.8)", width=0.8), name="Returns"), row=2, col=1)

fig.update_layout(title="SPY: Price and Returns",
                  height=550, plot_bgcolor="rgba(15,17,22,1)",
                  paper_bgcolor="rgba(15,17,22,1)", font=dict(color="white"))
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor="rgba(80,80,80,0.3)")
fig.show()


## 2. Forward Algorithm

The **forward variable** $\alpha_t(i)$ is the joint probability of observing the sequence up to time $t$ **and** being in state $S_i$ at time $t$:
$$\alpha_t(i) = P(O_1, O_2, \ldots, O_t,\, Q_t = S_i \mid \lambda)$$

### 2.1 Initialisation
$$\alpha_1(i) = \pi_i \cdot b_i(O_1), \quad i = 1,\ldots,N$$

### 2.2 Recursion 
$$\alpha_{t+1}(j) = \left[\sum_{i=1}^{N} \alpha_t(i)\, a_{ij}\right] b_j(O_{t+1}), \quad t = 1,\ldots,T-1$$

### 2.3 Termination
$$P(O \mid \lambda) = \sum_{i=1}^{N} \alpha_T(i)$$

Computational complexity: $O(N^2 T)$ -- polynomial in $T$, not exponential.

> **Implementation note:** we use *scaled* forward variables to prevent numerical underflow.  
> $\ln P(O|\lambda) = \sum_t \ln c_t$ where $c_t$ are the per-step normalisation constants.

In [ ]:
# Quick demonstration with manually set parameters (before training)
N_demo = 3
pi_demo = np.array([0.5, 0.3, 0.2])
A_demo  = np.array([[0.7, 0.2, 0.1],
                     [0.2, 0.6, 0.2],
                     [0.1, 0.3, 0.6]])
mu_demo    = np.array([-0.002, 0.001, -0.005])
sigma_demo = np.array([0.005,  0.012,  0.025])

alpha_demo, scales_demo = forward_pass(X_train[:50], pi_demo, A_demo, mu_demo, sigma_demo)

print(f"alpha shape  : {alpha_demo.shape}  (T=50, N=3)")
print(f"alpha[0]     : {alpha_demo[0].round(6)}  (sums to {alpha_demo[0].sum():.4f})")
print(f"alpha[-1]    : {alpha_demo[-1].round(6)}")
print(f"Log-likelihood (demo): {np.log(scales_demo).sum():.2f}")

## 3. Backward Algorithm

The **backward variable** $\beta_t(i)$ is the conditional probability of the *future* observations given state $S_i$ at time $t$:
$$\beta_t(i) = P(O_{t+1}, O_{t+2}, \ldots, O_T \mid Q_t = S_i,\, \lambda)$$

### 3.1 Initialisation 
$$\beta_T(i) = 1, \quad i = 1,\ldots,N$$

### 3.2 Recursion
$$\beta_t(i) = \sum_{j=1}^{N} a_{ij}\, b_j(O_{t+1})\, \beta_{t+1}(j), \quad t = T-1,\ldots,1$$

### 3.3 Key identities 
$$P(O\mid\lambda) = \sum_{i=1}^N \alpha_t(i)\,\beta_t(i) \quad \text{(at any } t \text{)}$$
$$\gamma_t(i) = P(Q_t=S_i \mid O,\lambda) = \frac{\alpha_t(i)\,\beta_t(i)}{\sum_k \alpha_t(k)\,\beta_t(k)}$$
$$\xi_t(i,j) = P(Q_t=S_i,\, Q_{t+1}=S_j \mid O,\lambda) = \frac{\alpha_t(i)\,a_{ij}\,b_j(O_{t+1})\,\beta_{t+1}(j)}{\sum_{i'}\sum_{j'} \alpha_t(i')\,a_{i'j'}\,b_{j'}(O_{t+1})\,\beta_{t+1}(j')}$$

In [ ]:
beta_demo = backward_pass(X_train[:50], A_demo, mu_demo, sigma_demo, scales_demo)
gamma_demo = compute_gamma(alpha_demo, beta_demo)

print(f"beta shape   : {beta_demo.shape}")
print(f"beta[-1]     : {beta_demo[-1].round(6)}  (initialized to 1 then scaled)")
print(f"gamma shape  : {gamma_demo.shape}")
print(f"gamma[0]     : {gamma_demo[0].round(4)}  (sums to {gamma_demo[0].sum():.4f})")

# Verify identity: gamma sums to 1 at every t
assert np.allclose(gamma_demo.sum(axis=1), 1.0, atol=1e-8), "gamma should sum to 1!"
print("\nGamma rows sum to 1 for all demo dates")

In [ ]:
fig = plot_alpha_beta(alpha_demo, beta_demo, max_t=50,
                      state_names=["S1","S2","S3"])
fig.show()

## 4. Viterbi Algorithm

Find the **most-probable hidden state sequence** $Q^* = \arg\max_Q P(Q \mid O, \lambda)$.

Define the Viterbi variable:
$$\delta_t(i) = \max_{Q_1,\ldots,Q_{t-1}} P(Q_1,\ldots,Q_t=S_i,\, O_1,\ldots,O_t \mid \lambda)$$

**Initialisation:**
$$\delta_1(i) = \pi_i \cdot b_i(O_1)$$

**Recursion** (in log-space to prevent underflow):
$$\log\delta_{t+1}(j) = \max_{i}\left[\log\delta_t(i) + \log a_{ij}\right] + \log b_j(O_{t+1})$$

Back-pointer:
$$\psi_t(j) = \arg\max_i\left[\delta_{t-1}(i)\cdot a_{ij}\right]$$

**Back-tracking** from $q_T^* = \arg\max_i \delta_T(i)$.

In [ ]:
# Viterbi on demo parameters
path_demo, log_prob_demo = viterbi(X_train[:100], pi_demo, A_demo, mu_demo, sigma_demo)
print(f"Viterbi path (first 20): {path_demo[:20]}")
print(f"Log-probability of best path: {log_prob_demo:.2f}")

## 5. Baum-Welch EM

The **Baum-Welch algorithm** is EM applied to HMMs:
$$\hat{\lambda} = \arg\max_\lambda P(O \mid \lambda)$$

### E-Step
Using current $\lambda^{(\text{old})}$, compute $\alpha_t(i)$, $\beta_t(i)$, then:
$$\gamma_t(i) = \frac{\alpha_t(i)\,\beta_t(i)}{\sum_k \alpha_t(k)\,\beta_t(k)}$$
$$\xi_t(i,j) = \frac{\alpha_t(i)\,a_{ij}\,b_j(O_{t+1})\,\beta_{t+1}(j)}{\sum_{i'}\sum_{j'}(\ldots)}$$

### M-Step
$$\pi_i^{\text{new}} = \gamma_1(i)$$
$$a_{ij}^{\text{new}} = \frac{\sum_{t=1}^{T-1} \xi_t(i,j)}{\sum_{t=1}^{T-1} \gamma_t(i)}$$
$$\mu_j^{\text{new}} = \frac{\sum_t \gamma_t(j)\,O_t}{\sum_t \gamma_t(j)}$$
$$\sigma_j^{2,\text{new}} = \frac{\sum_t \gamma_t(j)\,(O_t - \mu_j^{\text{new}})^2}{\sum_t \gamma_t(j)}$$

**Convergence:** stop when $|\mathcal{L}^{(k)} - \mathcal{L}^{(k-1)}| < \varepsilon$.

> **Proposition 5.1 (Monotone improvement):** each EM iteration guarantees $\mathcal{L}^{(k+1)} \geq \mathcal{L}^{(k)}$.

In [ ]:
# Train 3-state HMM from scratch
hmm = GaussianHMM(n_states=3, n_iter=300, tol=1e-7, random_state=42)
print("Training 3-state Gaussian HMM on SPY daily returns...")
hmm.fit(X_train)
print()
print(hmm.summary())


In [ ]:
# Quick checks after training.
# Numeric state ids are arbitrary, so I sort/interpret states by volatility later.
print("Transition row sums:", np.round(hmm.A_.sum(axis=1), 4))
print("State std devs      :", np.round(hmm.sigma_, 6))


In [ ]:
fig = plot_log_likelihood(hmm.log_likelihoods_,
                          title="Baum-Welch Convergence: 3-State HMM")
fig.show()

## 6. Regime Analysis & Results

In [ ]:
# Soft assignments (gamma) and hard assignments
gamma_train = hmm.predict_proba(X_train)
states_gamma = hmm.predict(X_train)
states_viterbi, log_p_viterbi = hmm.predict_viterbi(X_train)

state_names = ["Low-Vol", "Med-Vol", "High-Vol"]

print("Posterior argmax counts:", dict(zip(*np.unique(states_gamma, return_counts=True))))
print("Viterbi path counts     :", dict(zip(*np.unique(states_viterbi, return_counts=True))))
print("Smallest state share    :", f"{np.bincount(states_gamma).min() / len(states_gamma):.2%}")


In [ ]:
# State occupancy heat map over time
fig = plot_gamma(gamma_train, dates=df_train.index, state_names=state_names)
fig.show()

In [ ]:
# Assign regime labels to df_train
df_train = df_train.copy()
df_train["HMM_State"] = states_gamma

fig = plot_price_with_regimes(df_train, "HMM_State",
                               title="SPY: 3-State HMM States")
fig.show()

In [ ]:
fig = plot_regime_distributions(df_train, "HMM_State",
                                 title="HMM: Regime Return Distributions")
fig.show()

In [ ]:
fig = plot_transition_matrix(hmm.A_, state_names=state_names,
                             title="Learned Transition Matrix A")
fig.show()

In [ ]:
# Soft vs Viterbi comparison
df_vit = df_train.copy()
df_vit["Viterbi_State"] = states_viterbi

fig = plot_viterbi_path(df_vit, states_viterbi, state_names=state_names,
                         title="Viterbi Path: SPY")
fig.show()

In [ ]:
# Regime statistics
stats_hmm = regime_statistics(df_train["Returns"].values, states_gamma)
print(stats_hmm.round(4).to_string(index=False))

In [ ]:
persistence = regime_persistence_summary(states_gamma)
print(persistence.to_string(index=False))

In [ ]:
sharpes = conditional_sharpe(df_train["Returns"].values, states_gamma)
print("Conditional annualized Sharpe per state:")
for s, sh in sharpes.items():
    print(f"  State {s} ({state_names[s]}): {sh:.3f}")

## 7. Model Selection : Number of States N

Fit HMMs with N = 2, 3, 4, 5 states. Compare via AIC, BIC, and out-of-sample log-likelihood.

$$\text{AIC} = 2k - 2\mathcal{L}, \quad \text{BIC} = k \ln T - 2\mathcal{L}$$
where $k = N^2 + 2N$ is the number of free parameters.

In [ ]:
results = []
hmm_models = {}

# Fit HMMs with N = 2, 3, 4, 5 and compare train/test likelihood plus AIC/BIC
for N in range(2, 6):
    print(f"  Fitting {N}-state HMM...", end=" ")
    h = GaussianHMM(n_states=N, n_iter=300, tol=1e-7, random_state=42)
    h.fit(X_train)
    ll_train = h.log_likelihood(X_train)
    ll_test  = h.log_likelihood(X_test)
    k        = hmm_n_params(N)
    results.append({
        "N States": N,
        "Train LL":  round(ll_train, 1),
        "Test LL":   round(ll_test, 1),
        "AIC":       round(aic(ll_train, k), 1),
        "BIC":       round(bic(ll_train, k, len(X_train)), 1),
        "# Params":  k,
    })
    hmm_models[N] = h
    print(f"LL={ll_train:.1f}  AIC={aic(ll_train,k):.1f}  BIC={bic(ll_train,k,len(X_train)):.1f}")

print()
print(pd.DataFrame(results).to_string(index=False))

In [ ]:
from utils.viz_utils import plot_aic_bic

model_names = [f"{r['N States']}-state" for r in results]
aics = [r["AIC"] for r in results]
bics = [r["BIC"] for r in results]

fig = plot_aic_bic(model_names, aics, bics, title="HMM Model Selection: AIC vs BIC")
fig.show()

## 8. Out-of-Sample Evaluation

In [ ]:
df_test = df_test.copy()
df_test["HMM_State"] = hmm.predict(X_test)

fig = plot_price_with_regimes(df_test, "HMM_State",
                               title="SPY Test Set: HMM States")
fig.show()

stats_test = regime_statistics(df_test["Returns"].values, df_test["HMM_State"].values)
print("\nOut-of-sample regime statistics:")
print(stats_test.round(4).to_string(index=False))

## 9. Limitations

- Gaussian emissions are a simplification; financial returns are fat-tailed.
- State numbers can switch across runs, so I interpret states by volatility and return statistics.
- AIC/BIC may reward extra states that are not useful for explanation.
- This is a regime-detection demo, not a trading strategy by itself.


## 10. Summary

| Component | Implementation |
|---|---|
| Gaussian emission $b_j(O_t)$ | `hmm_core._emission_matrix` |
| Forward algorithm (scaled) | `hmm_core.forward_pass` |
| Backward algorithm (scaled) | `hmm_core.backward_pass` |
| State occupancy $\gamma_t(i)$ | `hmm_core.compute_gamma` |
| Transition occupancy $\xi_t(i,j)$ | `hmm_core.compute_xi` |
| Viterbi decoding | `hmm_core.viterbi` |
| Baum-Welch EM | `GaussianHMM.fit` |
| Model selection (AIC/BIC) | `utils.metrics` |